In [ ]:
!git clone https://github.com/d4-5/NLP4.git
%cd NLP4

In [ ]:
!pip install -r requirements.txt

In [ ]:
%cd NLP4

In [1]:
import os
import sys
import json
import pandas as pd
from pathlib import Path

sys.path.append(os.path.abspath("../"))

from src.agents import GroqLLMClient
from src.flow import NLPFlow
from src.llm_extract import DEFAULT_MODEL

In [2]:
with open("../docs/memory_policy_lab14.md", "r") as f:
    print(f.read())

# Політика пам'яті та знань — ЛР14

## 1. Пам'ять стану (Короткострокова)
Об'єкт `FlowState` слугує короткостроковою пам'яттю для одного виконання потоку NLP.

### Що зберігається:
- `case_id`: Унікальний ідентифікатор транзакції.
- `raw_input`: Оригінальний текст, наданий користувачем.
- `clean_text`: Очищена версія вхідних даних.
- `route`: Вибраний шлях пайплайну (наприклад, finance_extraction).
- `extraction_parsed`: JSON-вихід етапу виконання.
- `validation_issues`: Список помилок, знайдених валідатором або рецензентом.
- `fallback_output`: Результат механізму fallback, якщо він був активований.
- `status`: Поточний етап та індикатор успіху/відмови.
- `steps`: Аудиторський слід усіх переходів.

### Що НЕ зберігається:
- **API ключі / Облікові дані**: Ніколи не зберігаються у стані або логах.
- **Великі бінарні об'єкти**: Обробляються лише текстові дані.
- **PII (Персональні дані)**: Якщо це не потрібно для екстракції, їх слід мінімізувати або маскувати.
- **Чернетки з галюцинаціям

In [3]:
client = GroqLLMClient(model=DEFAULT_MODEL)
flow = NLPFlow(client)

In [4]:
test_cases = []
with open("../data/sample/lab14_test_cases.jsonl", "r") as f:
    for line in f:
        test_cases.append(json.loads(line))

print(f"Loaded {len(test_cases)} test cases.")

Loaded 10 test cases.


In [5]:
results = []
for case in test_cases:
    print(f"Processing: {case['input']}")
    result = flow.run(case['input'])
    results.append(result)
    print(f"Status: {result['status']}\n")

Processing: Переказати 500 грн на Monobank завтра
Status: exported_with_errors

Processing: Оплатити рахунок 12345
Status: exported

Processing: Сьогодні гарна погода для прогулянки
Status: exported

Processing: Відправ 1000 на карту 4444555566667777
Status: exported

Processing: Договір №987 від 12 травня 2024 року
Status: exported

Processing: Надіслати кошти без вказання суми
Status: exported_with_errors

Processing: !!!! @@@@ $$$$
Status: exported

Processing: Купити хліб за 25 грн та молоко за 40 грн
Status: exported

Processing:  
Status: failed

Processing: Переказ 100 USD на PayPal
Status: exported



In [6]:
total = len(results)
exported = len([r for r in results if "exported" in r['status']])

valid_no_fallback = len([r for r in results if not r.get('fallback_triggered', False) and r.get('status') == 'exported'])

fallback_triggered = len([r for r in results if r.get('fallback_triggered', False)])

fallback_success = len([r for r in results if r.get('fallback_triggered', False) and r.get('status') == 'exported'])

manual_review = len([r for r in results if r.get('needs_manual_review', False)])

metrics = {
    "Flow completion rate": f"{exported/total:.1%}" if total > 0 else "0%",
    "Validation pass rate": f"{valid_no_fallback/total:.1%}" if total > 0 else "0%",
    "Fallback activation rate": f"{fallback_triggered/total:.1%}" if total > 0 else "0%",
    "Fallback success rate": f"{fallback_success/fallback_triggered:.1%}" if fallback_triggered > 0 else "N/A",
    "Manual review / safe failure rate": f"{manual_review/total:.1%}" if total > 0 else "0%"
}

for k, v in metrics.items():
    print(f"{k}: {v}")

Flow completion rate: 90.0%
Validation pass rate: 0.0%
Fallback activation rate: 90.0%
Fallback success rate: 77.8%
Manual review / safe failure rate: 10.0%


In [7]:
analysis_df = pd.DataFrame(results)
analysis_df[['case_id', 'status', 'is_valid', 'needs_manual_review', 'errors', 'warnings']].to_csv("../docs/error_analysis_lab14.csv", index=False)
analysis_df

,case_id,timestamp,input,route,output,status,is_valid,fallback_triggered,fallback_method,warnings,errors,needs_manual_review
0,e3cccfdd,2026-05-19T15:51:57.094556,Переказати 500 грн на Monobank завтра,document_signal_schema,"{'document_id': None, 'document_type': None, '...",exported_with_errors,False,True,safe_failure,[],[Fallback failed to produce valid output],False
1,745ef9f0,2026-05-19T15:51:58.236124,Оплатити рахунок 12345,document_signal_schema,"{'document_id': '12345', 'document_type': 'GEN...",exported,True,True,repair_agent,[],[],False
2,7cf027da,2026-05-19T15:52:00.484267,Сьогодні гарна погода для прогулянки,generic_document_schema,"{'document_id': None, 'document_type': None, '...",exported,False,True,repair_agent,[],[],False
3,5b0ba31e,2026-05-19T15:52:01.547086,Відправ 1000 на карту 4444555566667777,document_signal_schema,"{'document_id': None, 'document_type': None, '...",exported,True,True,rule_based_partial,[Fallback success but needs manual review],[],False
4,23fd1b6a,2026-05-19T15:52:20.676041,Договір №987 від 12 травня 2024 року,document_signal_schema,"{'document_id': '987', 'document_type': 'CONTR...",exported,True,True,repair_agent,[],[],False
5,551b8097,2026-05-19T15:52:42.823915,Надіслати кошти без вказання суми,manual_review_route,"{'document_id': None, 'document_type': None, '...",exported_with_errors,True,True,safe_failure,[],[Fallback failed to produce valid output],False
6,dbe37ccd,2026-05-19T15:53:03.509542,!!!! @@@@ $$$$,document_signal_schema,"{'document_id': None, 'document_type': None, '...",exported,True,True,repair_agent,[],[],False
7,05602f42,2026-05-19T15:53:24.216865,Купити хліб за 25 грн та молоко за 40 грн,document_signal_schema,"{'document_id': None, 'document_type': None, '...",exported,True,True,repair_agent,[],[],False
8,99d2dff9,2026-05-19T15:53:46.675930,,NaN,None,failed,False,False,NaN,[],"[Empty input, Unexpected error: Empty input]",True
9,ccc60220,2026-05-19T15:53:46.676393,Переказ 100 USD на PayPal,document_signal_schema,"{'document_id': None, 'document_type': 'GENERI...",exported,False,True,repair_agent,[],[],False
